### Qdrant Hybrid Search with RRF

In [1]:
import requests
import uuid
import json
from uuid import uuid4

from qdrant_client import QdrantClient
from qdrant_client.http import models

from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings

In [2]:
client = QdrantClient("http://localhost:6333")
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='lmz-hybrid-search'), CollectionDescription(name='zoomcamp-faq'), CollectionDescription(name='zoomcamp-sparse-dense'), CollectionDescription(name='zoomcamp-sparse')])

In [3]:
url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'
docs_url = url_prefix + 'search_evaluation/documents-with-ids.json'
documents = requests.get(docs_url).json()

In [8]:
collection_name = "lmz-hybrid-search"
embedding_dim = 512
model_handle = "jinaai/jina-embeddings-v2-small-en"

In [9]:
client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "jina-small": models.VectorParams(
            size=embedding_dim,
            distance=models.Distance.COSINE
        )
    },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    }
)

True

In [11]:
points = []

for doc in documents:
    text = doc["question"] + " " + doc["text"]
    vector = {
        "jina-small": models.Document(text=doc["text"], model=model_handle),
        "bm25": models.Document(text=doc["text"], model="Qdrant/bm25")
    }
    point = models.PointStruct(
        id=uuid.uuid4().hex,
        vector=vector,
        payload=doc
    )
    points.append(point)

In [12]:
client.upsert(
    collection_name=collection_name,
    points=points
)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

tokenizer.json: 0.00B [00:00, ?B/s]

onnx/model.onnx:   0%|          | 0.00/130M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

norwegian.txt:   0%|          | 0.00/851 [00:00<?, ?B/s]

romanian.txt: 0.00B [00:00, ?B/s]

swedish.txt:   0%|          | 0.00/559 [00:00<?, ?B/s]

turkish.txt:   0%|          | 0.00/260 [00:00<?, ?B/s]

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [13]:
def rrf_search(query, limit=1):
    # Reciprocal Rank Fusion
    prefetch = [
        models.Prefetch(
            query=models.Document(
                text=query,
                model=model_handle
            ),
            using="jina-small",
            limit=(limit * 5)
        ),
        models.Prefetch(
            query=models.Document(
                text=query,
                model="Qdrant/bm25"
            ),
            using="bm25",
            limit=(limit * 5)
        )
    ]

    results = client.query_points(
        collection_name=collection_name,
        prefetch=prefetch,
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit,
        with_payload=True
    )

    return results.points

In [31]:
idx = 21
result = rrf_search(json.dumps(documents[idx]["question"]))
result[0].payload["text"], documents[idx]["text"]

('Using GitHub Codespaces in the browser resulted in a blank screen after the login to pgAdmin (running in a Docker container). The terminal of the pgAdmin container was showing the following error message:\nCSRFError: 400 Bad Request: The referrer does not match the host.\nSolution #1:\nAs recommended in the following issue  https://github.com/pgadmin-org/pgadmin4/issues/5432 setting the following environment variable solved it.\nPGADMIN_CONFIG_WTF_CSRF_ENABLED="False"\nModified “docker run” command\ndocker run --rm -it \\\n-e PGADMIN_DEFAULT_EMAIL="admin@admin.com" \\\n-e PGADMIN_DEFAULT_PASSWORD="root" \\\n-e PGADMIN_CONFIG_WTF_CSRF_ENABLED="False" \\\n-p "8080:80" \\\n--name pgadmin \\\n--network=pg-network \\\ndpage/pgadmin4:8.2\nSolution #2:\nUsing the local installed VSCode to display GitHub Codespaces.\nWhen using GitHub Codespaces in the locally installed VSCode (opening a Codespace or creating/starting one) this issue did not occur.',
 'GitHub Codespaces offers you computing 

### Langchain Qdrant

In [5]:
%%capture
collection_name = "lmz-hybrid-search-langchain"
embedding_dim = 512
model_handle = "jinaai/jina-embeddings-v2-small-en"

embeddings = FastEmbedEmbeddings(model_name=model_handle)
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

In [6]:
client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "jina-small": models.VectorParams(
            size=embedding_dim,
            distance=models.Distance.COSINE
        )
    },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    }
)

True

In [7]:
qdrant = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="jina-small",
    sparse_vector_name="bm25",
)

In [8]:
lc_docs = []
subset_docs = documents[:100]

for rd in subset_docs:
    doc = Document(
        page_content=rd["text"],
        metadata={
            "section": rd["section"],
            "course": rd["course"],
            "question": rd["question"]
        }
    )
    lc_docs.append(doc)

In [9]:
uuids = [str(uuid4()) for _ in range(len(lc_docs))]

qdrant.add_documents(documents=lc_docs, ids=uuids)

['bc2e7b27-7474-42fe-8428-a464deb7ebfa',
 'e02b15e0-f8b2-4477-b545-5e79003c85a1',
 '987b13ce-ffda-4849-9bb2-bf6b2cf3aa52',
 '70792ab8-5bc7-47fd-a447-3089668eebe1',
 '7874bcd7-6054-4540-b564-986b2ff5a547',
 '979c8027-b8c5-4a65-b6ed-a5ad6d1bc103',
 '563d2726-6879-461b-8778-3a00d91a461c',
 'b1141ca7-7ca0-47c9-84b8-c4a4d0d5dbef',
 'e0bfa160-5c86-47c8-bd0f-8ff05fd6031c',
 '7af6341d-bb2d-49fc-9e4a-e38adacf32d6',
 '0b8436b5-bd44-48a1-9d63-070ae6d8de29',
 '993dee2f-c71d-4f9d-9c5a-5469fc6b9477',
 'd51f0b7a-887d-4ff0-91bd-baac1c4c9fb4',
 '90434b9e-cff7-4f55-b39e-42082e91c140',
 '58628ecb-22fc-41ef-b579-a8088263e2a8',
 'cb2e9ec0-69ba-4d57-8cd6-7f33193de534',
 'c4643e05-95c4-4a2a-919c-78a9e513b2cb',
 '4084fb0a-87aa-4293-89d4-a2f63e5d8dd4',
 '5e53c588-353d-4f44-a673-8da0ab83b9d5',
 'f70e1e85-817f-4edd-a0e6-5f757dbee30d',
 'e8065954-3330-4ce7-9f38-9d28a16f7b08',
 '2bd6e066-a586-44a8-bb8a-ba21f9614350',
 '1c61da23-baa3-4266-9280-c5d53cee1fc4',
 'e6334bf1-6aca-490b-bd92-b87826cf75f8',
 'ba5398eb-7846-

In [11]:
query = subset_docs[21]["question"]
found_docs = qdrant.similarity_search(query)

In [13]:
found_docs[0].page_content

'GitHub Codespaces offers you computing Linux resources with many pre-installed tools (Docker, Docker Compose, Python).\nYou can also open any GitHub repository in a GitHub Codespace.'

In [15]:
documents[21]["text"]

'GitHub Codespaces offers you computing Linux resources with many pre-installed tools (Docker, Docker Compose, Python).\nYou can also open any GitHub repository in a GitHub Codespace.'